In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
#1. Load Data

df = pd.read_csv('/content/heart.csv') # Replace with your actual CSV file path
print("Original Data Shape:", df.shape)
print(df.head())



Original Data Shape: (303, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

   ca        thal  target  
0   0       fixed       0  
1   3      normal       1  
2   2  reversible       0  
3   0      normal       0  
4   0      normal       0  


In [ ]:
#2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())



Data Types:
 age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal         object
target        int64
dtype: object

Missing Values:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [ ]:
#3. Handle Missing Values

#Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

#Numerical: Replace with mean
num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

#Categorical: Replace with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])



In [ ]:
#4. Encode Categorical Variables

#Label Encoding (for binary categorical)
le = LabelEncoder()
for col in cat_cols:
 if df[col].nunique() == 2:
  df[col] = le.fit_transform(df[col])

# One-hot Encoding (for more than 2 categories)
df = pd.get_dummies(df, columns=[col for col in cat_cols if df[col].nunique() > 2], drop_first=True)


In [ ]:
#5. Outlier Detection and Treatment (using IQR)

for col in num_cols:
 Q1 = df[col].quantile(0.25)
 Q3 = df[col].quantile(0.75)
 IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df[col] = np.where(df[col] < lower, lower, df[col])
df[col] = np.where(df[col] > upper, upper, df[col])



In [ ]:
#6. Feature Scaling

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])


In [ ]:
target_col = 'target'  # Replace with your actual target column name

if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]

    bestfeatures = SelectKBest(score_func=f_classif, k=10)
    fit = bestfeatures.fit(X, y)

    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]

    print("\nSelected Features:", selected_features.to_list())
corr=df.corr()
corr.sort_values(by='target', ascending=False)







Selected Features: ['age', 'sex', 'cp', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal_normal', 'thal_reversible']


,age,sex,cp,thalach,exang,oldpeak,slope,ca,thal_normal,thal_reversible,target
target,0.198701,0.171564,0.381725,-0.386459,0.361026,0.475324,0.359572,0.476613,-0.461859,0.419345,1.000000
ca,0.368525,0.097593,0.190402,-0.270660,0.138042,0.298974,0.101242,1.000000,-0.260807,0.213829,0.476613
oldpeak,0.202158,0.117436,0.179624,-0.340969,0.278189,1.000000,0.566642,0.298974,-0.333423,0.299136,0.475324
thal_reversible,0.097981,0.322682,0.268109,-0.194723,0.296183,0.299136,0.221235,0.213829,-0.872485,1.000000,0.419345
cp,0.075133,0.011498,1.000000,-0.259882,0.337318,0.179624,0.185050,0.190402,-0.254149,0.268109,0.381725
exang,0.094736,0.135680,0.337318,-0.379367,1.000000,0.278189,0.245470,0.138042,-0.309924,0.296183,0.361026
slope,0.151392,0.033110,0.185050,-0.361260,0.245470,0.566642,1.000000,0.101242,-0.293358,0.221235,0.359572
age,1.000000,-0.090748,0.075133,-0.388988,0.094736,0.202158,0.151392,0.368525,-0.128209,0.097981,0.198701
sex,-0.090748,1.000000,0.011498,-0.070115,0.135680,0.117436,0.033110,0.097593,-0.378483,0.322682,0.171564
thalach,-0.388988,-0.070115,-0.259882,1.000000,-0.379367,-0.340969,-0.361260,-0.270660,0.272748,-0.194723,-0.386459


In [ ]:
import numpy as np
import sklearn.preprocessing as preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X = np.asarray(df[['age', 'sex', 'cp', 'thalach', 'exang', 'oldpeak', 'slope', 'ca','thal_normal', 'thal_reversible']])
y = np.asarray(df['target'])

X = preprocessing.StandardScaler().fit(X).transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 4)

print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)



Train set: (212, 10) (212,)
Test set: (91, 10) (91,)


In [ ]:
print(df.columns)

Index(['age', 'sex', 'cp', 'thalach', 'exang', 'oldpeak', 'slope', 'ca',
       'thal_normal', 'thal_reversible', 'target'],
      dtype='object')


In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.8571428571428571


In [ ]:
#2. K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.7802197802197802


In [ ]:
#3. Decision Tree
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.8021978021978022


In [ ]:
#4. Random Forest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.7912087912087912


In [ ]:
#5. Support Vector Machine
from sklearn.svm import SVC
model = SVC()
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.7912087912087912


In [ ]:
#6. Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier()
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.7912087912087912


In [ ]:
#7. Naive Bayes
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
# Convert y_train and y_test to integer type as classification models expect discrete labels
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test.astype(int)))

Accuracy: 0.8351648351648352
